# GoEmotions Model Inference Example (Baseline RNN)

This notebook demonstrates how to load a custom-trained Simple RNN model and perform inference on new text examples.

## Objectives
1. Download the trained model artefacts from Google Drive to `/tmp` 
2. Load the custom model
3. Define the `predict_emotion` function for multi-label inference
4. Test the model with example sentences

In [ ]:
import os
import re
import torch
import torch.nn as nn
import numpy as np
import json
from typing import List

# Constants
MODEL_PATH = '/tmp/model.pt'
VOCAB_PATH = '/tmp/vocab.json' # You should save your word_to_index mapping during training
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hardcoded labels from GoEmotions dataset
EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring", 
    "confusion", "curiosity", "desire", "disappointment", "disapproval", 
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief", 
    "joy", "love", "nervousness", "optimism", "pride", "realization", 
    "relief", "remorse", "sadness", "surprise", "neutral"
]
num_labels = len(EMOTION_LABELS)

print(f"Using device: {DEVICE}")

## 1. Download Model Artifacts

Use `gdown` to download your trained model weights and vocabulary from Google Drive.

In [ ]:
def download_from_drive():
    """
    Downloads the model and vocab from Google Drive.
    """
    # TODO: Change me - Replace with your actual Google Drive file IDs
    # !pip install gdown
    # import gdown
    
    # model_file_id = 'YOUR_MODEL_FILE_ID'
    # vocab_file_id = 'YOUR_VOCAB_FILE_ID'
    
    # gdown.download(f'https://drive.google.com/uc?id={model_file_id}', MODEL_PATH, quiet=False)
    # gdown.download(f'https://drive.google.com/uc?id={vocab_file_id}', VOCAB_PATH, quiet=False)
    
    print("TODO: Please update the file IDs above to download your model artifacts.")

download_from_drive()

## 2. Model Definition & Loading

Defining the architecture of our baseline model. Make sure that you have stored your model and volabulary or tokenizer before executing this notebook

In [ ]:
class SimpleRNNBaseline(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        # We take the last hidden state
        return self.fc(hidden.squeeze(0))

def load_model_and_vocab():
    # Placeholder vocab for initialization if files don't exist yet
    if not os.path.exists(VOCAB_PATH):
        print("Vocab file not found. Using dummy vocab size.")
        vocab = {"<PAD>": 0, "<UNK>": 1}
    else:
        with open(VOCAB_PATH, 'r') as f:
            vocab = json.load(f)
    
    vocab_size = len(vocab)
    model = SimpleRNNBaseline(vocab_size, 100, 128, num_labels).to(DEVICE)
    
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print("Model weights loaded successfully.")
    else:
        print(f"Model weights not found at {MODEL_PATH}. Using randomly initialized weights.")
        
    model.eval()
    return model, vocab

model, vocab = load_model_and_vocab()

## 3. Inference Function

The `predict_emotion` function handles tokenization and thresholding.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

def predict_emotion(text: str, threshold: float = 0.5) -> List[str]:
    """
    Predicts the emotions expressed in a given text.

    Args:
        text (str): The input text to be evaluated.
        threshold (float, optional): Minimum probability required for an emotion
            to be included in the output. Defaults to 0.5.

    Returns:
        List[str]: A list of emotion labels whose predicted probabilities exceed
        the given threshold, sorted in alphabetical order.
    """
    # Tokenization
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab.get("<UNK>", 1)) for t in tokens]
    if not indices: indices = [vocab.get("<UNK>", 1)]
    
    # Prepare input tensor
    x = torch.tensor([indices], dtype=torch.long).to(DEVICE)
    
    # Inference
    with torch.no_grad():
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    # Find all labels above threshold
    predicted_emotions = [
        EMOTION_LABELS[i] for i, prob in enumerate(probs) if prob >= threshold
    ]
    
        
    return sorted(predicted_emotions)

# Quick test
test_text = "I am so happy that everything worked out!"
print(f"Text: {test_text}")
print(f"Predicted Emotions: {predict_emotion(test_text)}")

## 4. Example Predictions

Testing with various examples, including the required one.

In [ ]:
examples = [
    "This program terrible. I will never watch it again",
    "I am so grateful for your help!",
    "I don't know what to do next, I'm a bit lost.",
    "That's hilarious! I can't stop laughing.",
    "I'm really worried about the upcoming exam."
]

print(f"{'-'*50}")
for text in examples:
    emotions = predict_emotion(text)
    print(f"Text: {text}")
    print(f"Emotions: {', '.join(emotions)}")
    print(f"{'-'*50}")